# AI PDF to CSV Automation
### Research experiment + interactive invoice extraction application

This notebook compares **traditional Python extraction** with **Qwen 2.5 1.5B Instruct** on PDF invoices, validates extracted fields, benchmarks processing time, and provides a Gradio interface for uploading a PDF and downloading the extracted CSV.

**Important scope notes**
- The research sample uses five synthetic, text-based invoices.
- The application processes text-based PDF pages. Scanned/image-only PDFs need OCR, which is not included in this version.
- One invoice per PDF page is assumed.
- Financial totals are read from explicit total labels (for example, `Total Amount Due` or `Grand Total Due`) rather than inferred from quantity × unit price.
- Run cells from top to bottom in Google Colab. The Gradio launch cell intentionally stays running while the app is available.

## 1. Install dependencies

Run this cell once per fresh Colab runtime.

In [ ]:
!pip -q install pymupdf pandas matplotlib transformers accelerate gradio

## 2. Import libraries and configure the runtime

In [ ]:
import os
import re
import json
import time
import tempfile
import warnings

import fitz
import pandas as pd
import matplotlib.pyplot as plt
import torch
import gradio as gr

from transformers import AutoTokenizer, AutoModelForCausalLM

pd.set_option("display.max_columns", None)
warnings.filterwarnings("ignore")

print("Libraries imported successfully.")
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3. Load the Qwen model

The model is loaded once and reused by both the research experiment and the Gradio application. A Colab GPU runtime (T4 or better) is recommended.

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)
model.eval()

print("Model loaded successfully.")
print("Model device:", model.device)

## 4. Upload the five research invoices

Upload `sample_invoices_for_pdf_to_csv_project.pdf` (or your copy of the five-invoice sample PDF). The notebook will process each page as one invoice.

In [ ]:
from google.colab import files

uploaded = files.upload()
if not uploaded:
    raise ValueError("No PDF was uploaded.")

research_pdf_path = next(iter(uploaded.keys()))
research_document = fitz.open(research_pdf_path)

print("PDF:", research_pdf_path)
print("Pages / invoices:", len(research_document))

## 5. Shared extraction utilities

The total amount is obtained from the PDF text around a final-total label. This avoids the exact issue encountered during testing, where the AI selected the line subtotal (`52,500`) instead of the grand total due (`61,950`).

In [ ]:
FIELDS = [
    "Invoice_ID", "Invoice_Date", "Customer", "Product",
    "Quantity", "Unit_Price", "Total_Amount"
]

def parse_number(value):
    """Convert a currency-like string to float; return None when unavailable."""
    if value is None:
        return None
    cleaned = re.sub(r"[^0-9.\-]", "", str(value).replace(",", ""))
    if not cleaned or cleaned in {".", "-", "-."}:
        return None
    try:
        return float(cleaned)
    except ValueError:
        return None


def extract_labeled_amount(invoice_text):
    """
    Find an explicitly labelled final invoice amount.
    Supports labels such as Grand Total Due, Total Amount Due,
    Amount Payable and Invoice Total. Reads the amount on the same
    line or the next non-empty line.
    """
    lines = [line.strip() for line in invoice_text.splitlines() if line.strip()]
    label_pattern = re.compile(
        r"(grand\s+total\s+due|total\s+amount\s+due|amount\s+payable|"
        r"total\s+payable|invoice\s+total|total\s+due)",
        re.IGNORECASE
    )

    # Prefer the most explicit final-total labels.
    for preferred in [r"grand\s+total\s+due", r"total\s+amount\s+due",
                      r"amount\s+payable", r"total\s+payable",
                      r"invoice\s+total", r"total\s+due"]:
        pattern = re.compile(preferred, re.IGNORECASE)
        for i, line in enumerate(lines):
            if pattern.search(line):
                # First try amount on the same line after the label.
                tail = pattern.sub("", line).strip(" :-")
                amount = parse_number(tail)
                if amount is not None:
                    return amount
                # Then inspect the next two non-empty lines.
                for candidate in lines[i + 1:i + 3]:
                    amount = parse_number(candidate)
                    if amount is not None:
                        return amount
    return None


def extract_invoice_id(text):
    patterns = [
        r"(?:Invoice\s*(?:ID|No\.?|Number|#)|Invoice\s*Reference)\s*[:\-]?\s*([A-Z0-9][A-Z0-9\-\/]+)",
    ]
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(1).strip()

    # Some invoice layouts put the label and value on separate lines.
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    for i, line in enumerate(lines):
        if re.fullmatch(
            r"(?:INVOICE\s*(?:ID|NO\.?|NUMBER|#)|INVOICE\s*REFERENCE)",
            line, re.IGNORECASE
        ) and i + 1 < len(lines):
            return lines[i + 1].strip()
    return None


def extract_invoice_date(text):
    match = re.search(
        r"Invoice\s*Date\s*[:\-]?\s*([^\n]+)",
        text, re.IGNORECASE
    )
    if match:
        value = match.group(1).strip()
        if value:
            return value
        lines = [x.strip() for x in text.splitlines() if x.strip()]
        for i, line in enumerate(lines):
            if re.fullmatch(r"Invoice\s*Date", line, re.IGNORECASE) and i + 1 < len(lines):
                return lines[i + 1]
    lines = [x.strip() for x in text.splitlines() if x.strip()]
    for i, line in enumerate(lines):
        if re.fullmatch(r"Invoice\s*Date", line, re.IGNORECASE) and i + 1 < len(lines):
            return lines[i + 1]
    return None


def extract_customer(text):
    lines = [x.strip() for x in text.splitlines() if x.strip()]
    for i, line in enumerate(lines):
        if re.fullmatch(r"(Bill\s*To|Customer|Billed\s*To)", line, re.IGNORECASE):
            if i + 1 < len(lines):
                return lines[i + 1]
    match = re.search(r"(?:Bill\s*To|Customer)\s*[:\-]\s*([^\n]+)", text, re.IGNORECASE)
    return match.group(1).strip() if match else None


def extract_traditional_invoice(page):
    """
    Traditional rule-based baseline for the structured research invoices.
    It uses labels and the known tabular text order; it is not intended
    to handle every possible vendor invoice layout.
    """
    text = page.get_text("text")
    lines = [x.strip() for x in text.splitlines() if x.strip()]

    invoice_id = extract_invoice_id(text)
    invoice_date = extract_invoice_date(text)
    customer = extract_customer(text)
    total_amount = extract_labeled_amount(text)

    product = quantity = unit_price = None

    # The provided five-invoice research PDF places the item row after
    # the "Line Subtotal (INR)" table header.
    header_index = next(
        (i for i, line in enumerate(lines)
         if re.search(r"Line\s+Subtotal", line, re.IGNORECASE)),
        None
    )
    if header_index is not None and header_index + 4 < len(lines):
        product = lines[header_index + 1]
        quantity = parse_number(lines[header_index + 2])
        unit_price = parse_number(lines[header_index + 3])

    return {
        "Invoice_ID": invoice_id,
        "Invoice_Date": invoice_date,
        "Customer": customer,
        "Product": product,
        "Quantity": quantity,
        "Unit_Price": unit_price,
        "Total_Amount": total_amount
    }


def normalize_ai_record(data):
    """Flatten common nested AI values and normalize the seven output fields."""
    def flatten(value, preferred_keys):
        if isinstance(value, dict):
            for key in preferred_keys:
                if key in value:
                    return flatten(value[key], preferred_keys)
            return None
        if isinstance(value, list):
            if not value:
                return None
            return flatten(value[0], preferred_keys)
        return value

    result = {field: flatten(data.get(field), ["Name", "name", "Description",
                                               "description", "value", "text"])
              for field in FIELDS}
    for field in ["Quantity", "Unit_Price", "Total_Amount"]:
        result[field] = parse_number(result[field])
    for field in ["Invoice_ID", "Invoice_Date", "Customer", "Product"]:
        if result[field] is not None:
            result[field] = str(result[field]).strip()
    return result

## 6. AI extraction function

Qwen extracts the descriptive and numeric fields. The final total is then replaced with the amount found beside an explicit final-total label in the PDF text, when such a label is present. This is a deterministic correction, not a hardcoded invoice value.

In [ ]:
def extract_invoice_with_ai(page):
    invoice_text = page.get_text("text")
    if not invoice_text.strip():
        raise ValueError(
            "This page has no selectable text. Scanned PDFs require OCR."
        )

    prompt = f"""
You are an invoice data extraction assistant.
Extract these fields from the invoice:
Invoice_ID, Invoice_Date, Customer, Product, Quantity, Unit_Price, Total_Amount.

Rules:
- Customer: the bill-to customer name only.
- Product: the product description only.
- Quantity: numeric quantity.
- Unit_Price: numeric price for one unit.
- Total_Amount: final amount payable including taxes, not the subtotal
  or product line amount. Prefer labels such as Grand Total Due,
  Total Amount Due, Amount Payable or Total Payable.
- Return one flat JSON object only. No Markdown fences or explanation.
- Use null if a field is genuinely absent. Do not invent values.
- Numeric fields must be numbers without currency symbols.

Example:
{{
  "Invoice_ID": "INV-1001",
  "Invoice_Date": "20 November 2026",
  "Customer": "ABC Learning Services",
  "Product": "Office notebooks",
  "Quantity": 50,
  "Unit_Price": 120.00,
  "Total_Amount": 7080.00
}}

Invoice text:
{invoice_text}
"""

    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=350,
            do_sample=False
        )

    response = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    # Remove optional fences and isolate the first JSON object.
    response = re.sub(r"```(?:json)?|```", "", response,
                      flags=re.IGNORECASE).strip()
    start, end = response.find("{"), response.rfind("}") + 1
    if start < 0 or end <= start:
        raise ValueError(f"AI did not return a JSON object. Response: {response[:500]}")

    try:
        raw_data = json.loads(response[start:end])
    except json.JSONDecodeError as exc:
        raise ValueError(
            f"AI returned malformed JSON: {exc}. Response: {response[:500]}"
        ) from exc

    record = normalize_ai_record(raw_data)

    # Reliable final-total extraction from the source PDF text.
    labelled_total = extract_labeled_amount(invoice_text)
    if labelled_total is not None:
        record["Total_Amount"] = labelled_total

    # Fill the invoice ID/date/customer/total from labelled source text
    # if the model missed one of these straightforward fields.
    record["Invoice_ID"] = record["Invoice_ID"] or extract_invoice_id(invoice_text)
    record["Invoice_Date"] = record["Invoice_Date"] or extract_invoice_date(invoice_text)
    record["Customer"] = record["Customer"] or extract_customer(invoice_text)

    return record

## 7. Run both extraction methods on the research PDF

This section rebuilds both datasets from the uploaded five-page PDF.

In [ ]:
traditional_records = []
ai_records = []

for page_number, page in enumerate(research_document, start=1):
    print(f"Processing research invoice {page_number}/{len(research_document)}...")
    traditional_records.append(extract_traditional_invoice(page))
    ai_records.append(extract_invoice_with_ai(page))

traditional_df = pd.DataFrame(traditional_records, columns=FIELDS)
ai_df = pd.DataFrame(ai_records, columns=FIELDS)

print("Traditional extraction:")
display(traditional_df)

print("AI extraction:")
display(ai_df)

## 8. Data quality checks

In [ ]:
def data_quality_report(df, method_name):
    print(f"DATA QUALITY: {method_name}")
    print("-" * 50)
    print("Records:", len(df))
    print("\nMissing values:")
    display(df.isnull().sum().to_frame("Missing Count"))
    print("Duplicate invoice IDs:", df["Invoice_ID"].dropna().duplicated().sum())
    print("Unique invoice IDs:", df["Invoice_ID"].nunique())
    print("\nData types:")
    display(df.dtypes.to_frame("Data Type"))

data_quality_report(traditional_df, "Traditional Python")
data_quality_report(ai_df, "AI Extraction")

## 9. Field-level comparison and accuracy

In [ ]:
comparison_df = traditional_df.merge(
    ai_df, on="Invoice_ID", how="outer",
    suffixes=("_Traditional", "_AI")
)

comparison_fields = [
    "Invoice_Date", "Customer", "Product",
    "Quantity", "Unit_Price", "Total_Amount"
]

for field in comparison_fields:
    left = comparison_df[f"{field}_Traditional"].fillna("").astype(str).str.strip()
    right = comparison_df[f"{field}_AI"].fillna("").astype(str).str.strip()
    # Compare numeric values with a small tolerance.
    if field in ["Quantity", "Unit_Price", "Total_Amount"]:
        left_num = pd.to_numeric(comparison_df[f"{field}_Traditional"], errors="coerce")
        right_num = pd.to_numeric(comparison_df[f"{field}_AI"], errors="coerce")
        comparison_df[f"{field}_Match"] = (
            left_num.notna() & right_num.notna() &
            ((left_num - right_num).abs() < 0.01)
        )
    else:
        comparison_df[f"{field}_Match"] = left.str.casefold() == right.str.casefold()

match_columns = ["Invoice_ID"] + [f"{field}_Match" for field in comparison_fields]
display(comparison_df[match_columns])

total_checks = len(comparison_df) * len(comparison_fields)
successful_checks = int(
    comparison_df[[f"{field}_Match" for field in comparison_fields]].sum().sum()
)
field_accuracy = (successful_checks / total_checks * 100) if total_checks else 0

print("Total field comparisons:", total_checks)
print("Successful matches:", successful_checks)
print("Mismatches:", total_checks - successful_checks)
print("Field-level match percentage:", round(field_accuracy, 2), "%")

## 10. Financial validation

For the provided five-invoice sample, compare the AI final amount against the total obtained independently by the traditional text-label parser. Do not calculate a grand total by simply multiplying quantity and unit price, because tax and other charges may apply.

In [ ]:
financial_validation = pd.DataFrame({
    "Invoice_ID": traditional_df["Invoice_ID"],
    "Traditional_Total": traditional_df["Total_Amount"],
    "AI_Total": ai_df["Total_Amount"]
})
financial_validation["Amount_Match"] = (
    pd.to_numeric(financial_validation["Traditional_Total"], errors="coerce").notna()
    & pd.to_numeric(financial_validation["AI_Total"], errors="coerce").notna()
    & (
        pd.to_numeric(financial_validation["Traditional_Total"], errors="coerce")
        - pd.to_numeric(financial_validation["AI_Total"], errors="coerce")
    ).abs().lt(0.01)
)
display(financial_validation)

print("Financial validations passed:",
      int(financial_validation["Amount_Match"].sum()), "/", len(financial_validation))
print("Traditional total revenue:",
      pd.to_numeric(traditional_df["Total_Amount"], errors="coerce").sum())
print("AI total revenue:",
      pd.to_numeric(ai_df["Total_Amount"], errors="coerce").sum())

## 11. Performance benchmark

In [ ]:
# Warm-up one short model call before timing to reduce first-call overhead.
_ = extract_invoice_with_ai(research_document[0])

traditional_times = []
ai_times = []
BENCHMARK_RUNS = 3

for run in range(BENCHMARK_RUNS):
    start = time.perf_counter()
    _ = [extract_traditional_invoice(page) for page in research_document]
    traditional_times.append(time.perf_counter() - start)

for run in range(BENCHMARK_RUNS):
    start = time.perf_counter()
    _ = [extract_invoice_with_ai(page) for page in research_document]
    ai_times.append(time.perf_counter() - start)

benchmark_df = pd.DataFrame({
    "Method": ["Traditional Python", "AI Extraction"],
    "Average Time (seconds)": [
        sum(traditional_times) / len(traditional_times),
        sum(ai_times) / len(ai_times)
    ],
    "Runs": [BENCHMARK_RUNS, BENCHMARK_RUNS]
})
display(benchmark_df)

traditional_avg = benchmark_df.loc[0, "Average Time (seconds)"]
ai_avg = benchmark_df.loc[1, "Average Time (seconds)"]
print("AI / Traditional time ratio:",
      round(ai_avg / traditional_avg, 2) if traditional_avg else "N/A")

## 12. Visualize performance

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(benchmark_df["Method"], benchmark_df["Average Time (seconds)"])
plt.title("Average Invoice Extraction Time")
plt.ylabel("Seconds (lower is faster)")
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

## 13. Research findings and limitations

Use the values generated above for the final report after running the notebook. The previous notebook documented a five-invoice result of 100% field matches, with traditional extraction averaging approximately 0.0032 seconds and AI extraction averaging approximately 17.2741 seconds. Those are historical recorded measurements; the benchmark cell above recalculates the timings in the current runtime.

**Limitations:** the sample is small and synthetic; the traditional baseline is tailored to the sample's structured layout; AI extraction can vary; and scanned PDFs are not supported without OCR. Test additional invoice layouts before making broad accuracy claims.

## 14. Export research outputs

In [ ]:
output_dir = "research_results"
os.makedirs(output_dir, exist_ok=True)

traditional_df.to_csv(os.path.join(output_dir, "Traditional_Invoice_Extraction.csv"), index=False)
ai_df.to_csv(os.path.join(output_dir, "AI_Invoice_Extraction.csv"), index=False)
comparison_df.to_csv(os.path.join(output_dir, "Field_Level_Comparison.csv"), index=False)
financial_validation.to_csv(os.path.join(output_dir, "Financial_Validation.csv"), index=False)
benchmark_df.to_csv(os.path.join(output_dir, "Performance_Benchmark.csv"), index=False)

print("Research outputs saved in:", output_dir)
print("\nFiles:")
for name in sorted(os.listdir(output_dir)):
    print("-", name)

# PART B — Interactive Gradio Application

## 15. Application processing function

The app accepts one PDF at a time and treats each page as one invoice. It uses Qwen for extraction, applies the same source-text final-total correction, runs basic quality checks, and creates a downloadable CSV.

In [ ]:
def process_uploaded_invoice(pdf_path):
    if not pdf_path:
        raise gr.Error("Please upload a PDF invoice.")

    records = []
    try:
        pdf = fitz.open(pdf_path)
    except Exception as exc:
        raise gr.Error(f"Could not open the uploaded PDF: {exc}")

    try:
        if len(pdf) == 0:
            raise gr.Error("The uploaded PDF contains no pages.")

        for page_number, page in enumerate(pdf, start=1):
            try:
                record = extract_invoice_with_ai(page)
                record["Page"] = page_number
                records.append(record)
            except Exception as exc:
                raise gr.Error(f"Extraction failed on page {page_number}: {exc}")
    finally:
        pdf.close()

    result_df = pd.DataFrame(records)
    result_df["Missing_Fields"] = result_df[FIELDS].isnull().sum(axis=1)
    result_df["Duplicate_Invoice_ID"] = (
        result_df["Invoice_ID"].notna()
        & result_df["Invoice_ID"].duplicated(keep=False)
    )

    missing_total = int(result_df[FIELDS].isnull().sum().sum())
    duplicate_ids = int(result_df["Invoice_ID"].dropna().duplicated().sum())
    total_amount = pd.to_numeric(result_df["Total_Amount"], errors="coerce").sum()

    summary = (
        "PROCESSING COMPLETED\n"
        f"Pages processed: {len(result_df)}\n"
        f"Missing field values: {missing_total}\n"
        f"Duplicate invoice IDs: {duplicate_ids}\n"
        f"Extracted final total across pages: {total_amount:,.2f}\n\n"
        "Note: The final amount is taken from an explicit total label in the PDF "
        "when available. Scanned/image-only PDFs require OCR."
    )

    csv_path = os.path.join(
        tempfile.gettempdir(),
        f"AI_Invoice_Extraction_{int(time.time())}.csv"
    )
    result_df.to_csv(csv_path, index=False)

    return result_df, summary, csv_path

## 16. Build the Gradio user interface

Upload a PDF, click **Extract Invoice Data**, review the table and validation summary, then download the CSV.

In [ ]:
with gr.Blocks(title="AI PDF to CSV Automation", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "# AI PDF to CSV Automation\n"
        "Upload a text-based invoice PDF. The Qwen model extracts invoice fields, "
        "checks basic data quality, and generates a CSV file."
    )

    pdf_input = gr.File(
        label="Upload invoice PDF",
        file_types=[".pdf"],
        type="filepath"
    )
    extract_button = gr.Button("Extract Invoice Data", variant="primary")

    extracted_table = gr.Dataframe(
        label="Extracted Invoice Data",
        interactive=False,
        wrap=True
    )
    status_output = gr.Textbox(
        label="Processing and Validation Summary",
        lines=7,
        interactive=False
    )
    csv_download = gr.File(label="Download CSV")

    extract_button.click(
        fn=process_uploaded_invoice,
        inputs=pdf_input,
        outputs=[extracted_table, status_output, csv_download]
    )

print("Gradio interface created successfully.")

## 17. Launch the application

**Important:** this cell is expected to remain running while the Gradio app is online. That is normal; it is serving the interface, not stuck processing. Open the public Gradio link displayed in the output. Stop this cell only when you want to shut down the app.

In [ ]:
demo.launch(share=True, debug=True)